# Falling body with drag — identifiability at 3 unknowns

The `falling-body` branch of **systemid**. Same gray-box split as the CADAC work,
at a size where the true matrix is known in closed form:

```
state  x = [y, v]
ydot = v
vdot = -g  -  (k/m)|v| v        <- only this is learned

    A = [ 0        1        ]      c = [  0  ]
        [ 0   -(k/m)|v|     ]          [ -g  ]
```

Free after masking: `dA[1,0]` (true **0**), `dA[1,1]` (true `-(k/m)|v|`),
`dc[1]` (true **0**). Three unknowns, one equation.

No CADAC, no compilation, no Drive. Data generates in under a minute.

**Measured on the Pi at λ=0** (the result this notebook is here to extend):
prediction error 0.0060 m/s² against a 9.37 m/s² signal, and *52% of the force*
in blocks whose true value is exactly zero. Sections 5–7 ask whether λ fixes it.


## 1 · Environment


In [ ]:
import subprocess, sys, torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print(sys.version)


## 2 · Clone the branch

Unconditional `fetch` + `reset --hard`, and `sys.modules` is purged: a runtime that
cloned in an earlier session otherwise keeps running old code while the notebook is
current, and nothing says so. That trap cost a full 50-run generation on `main`.


In [ ]:
BRANCH = 'falling-body'  # @param {type:"string"}
REPO   = 'https://github.com/alican30alicanexe-alt/systemid.git'

from pathlib import Path
CODE = Path('/content/systemid')

if not (CODE / '.git').exists():
    subprocess.run(['git', 'clone', REPO, str(CODE)], check=True)
subprocess.run(['git', '-C', str(CODE), 'fetch', '--all', '--prune'], check=True)
subprocess.run(['git', '-C', str(CODE), 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', str(CODE), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
print(subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--oneline'],
                     capture_output=True, text=True).stdout)

if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

# A refresh achieves nothing if the stale module objects survive it.
for name in ['fall', 'physics', 'model', 'dataset', 'trainer', 'evaluate',
             'identifiability', 'generator']:
    sys.modules.pop(name, None)

import fall

# Assert the schema at the point of import, not at the point of use.
assert fall.DEFAULT_STATE == ('y', 'v'), fall.DEFAULT_STATE
assert fall.DEFAULT_PARAMS == ('mass', 'speed'), fall.DEFAULT_PARAMS
assert set(fall.MODULES) == {'kinematics', 'gravity', 'drag'}, sorted(fall.MODULES)
print('states:', fall.DEFAULT_STATE)
print('params:', fall.DEFAULT_PARAMS)
print('k =', fall.K_DRAG, 'kg/m   g =', fall.G, 'm/s^2')


## 3 · Generate data, and run the gate

`residual_report` prints two things. The residual must grow with `|v|` — the same
shape as `main`'s `pdynmc` gate. And it must equal `-(k/m)|v|v` **exactly**, which
is the check CADAC cannot offer: there, "the residual is aerodynamics" rested on
`FSPB`, an output of the same simulator.

`GATE_OK` gates section 5. Do not train through a red gate — a wrong analytical
module is learned as drag, which is the quantity being measured.


In [ ]:
N_RUNS = 50      # @param {type:"integer"}
SEED   = 0       # @param {type:"integer"}
DATA   = Path('/content/fall.npz')

fall.generate(N_RUNS, DATA, seed=SEED)
print()
residual = fall.residual_report(DATA)

import numpy as np, torch
d = np.load(DATA, allow_pickle=True)
x, p = torch.tensor(d['x']), torch.tensor(d['p'])
closed_form_error = float((residual[:, 1] - fall.drag_acceleration(x, p[:, 0])).abs().max())
kinematic_error   = float(residual[:, 0].abs().max())

GATE_OK = closed_form_error < 1e-9 and kinematic_error < 1e-9
print(f"\nclosed form  max error {closed_form_error:.3e} m/s^2")
print(f"kinematic    max error {kinematic_error:.3e} m/s")
print('GATE OK' if GATE_OK else 'GATE FAILED -- do not train')


## 4 · Configuration

`LAMBDA_REG` is the whole experiment. At 0 the SDC factorisation is non-unique and
the recovered matrix is arbitrary among equally-good solutions; raising it selects
the minimum-norm correction. On the CADAC dataset λ=0.1 cut matrix disagreement
7.7× for 16% accuracy. Whether it moves the force into the *right* block is what
only this system can answer.


In [ ]:
LAMBDA_REG = 0.1     # @param {type:"number"}
EPOCHS     = 200     # @param {type:"integer"}
BATCH      = 2048    # @param {type:"integer"}
LR         = 2e-3    # @param {type:"number"}
HIDDEN     = [128, 128]
RUN_NAME   = f'fall_lam{LAMBDA_REG:g}'

from dataset import build_loaders
from model import GrayBoxSSM
from physics import StateLayout
from trainer import TrainConfig, Trainer

torch.manual_seed(SEED)
train_loader, val_loader, test, meta = build_loaders(DATA, batch_size=BATCH, seed=SEED)
layout  = StateLayout(test.state_names, test.param_names)
physics = fall.make_physics(layout)
vel     = slice(layout.s('v'), layout.s('v') + 1)
print(physics)
print('free dA entries:', int(physics.free_mask().sum()), '+ dc offsets')


## 5 · Train


In [ ]:
if not GATE_OK:
    raise RuntimeError('gate failed in section 3 -- fix the physics before training')

model = GrayBoxSSM.from_data(train_loader, physics,
                             n_param=len(test.param_names), hidden=HIDDEN)
print(f'[model] {sum(q.numel() for q in model.parameters())} parameters')

cfg = TrainConfig(epochs=EPOCHS, lr=LR, lambda_reg=LAMBDA_REG,
                  ckpt_dir=Path('/content/checkpoints'), run_name=RUN_NAME,
                  device='cuda' if torch.cuda.is_available() else 'cpu')
trainer = Trainer(model, cfg, q_index=layout.p('speed'), vel_slice=vel,
                  buckets=fall.V_BUCKETS)
history = trainer.fit(train_loader, val_loader)


## 6 · Where did the force go?

**This is the cell the branch exists for.** `main` could print this decomposition
but not check it; here every entry has a closed form.

Reference, measured at λ=0 on the same dataset:

| | λ=0 | truth |
|---|---|---|
| `dA[1,0]·y` | 1.1379 | **0** |
| `dA[1,1]·v` | 3.6560 | 7.5428 |
| `dc[1]` | 2.7811 | **0** |
| `dA[1,1]` rel error at \|v\|>40 | **50.2%** | — |


In [ ]:
model.to('cpu')
trainer.load_checkpoint()
model.to('cpu')
stats = fall.matrix_report(model, test.x, test.p)

share = stats['a_vel'] / max(stats['a_pos'] + stats['a_vel'] + stats['a_off'], 1e-12)
print(f"\ndrag block carries {100 * share:.1f}% of the identified force "
      f"(should be 100%)")
print(f"dA[1,1] relative error: {100 * stats['coeff_rel']:.2f}%")
print('VERDICT:', 'READABLE' if share > 0.9 and stats['coeff_rel'] < 0.1
      else 'MISATTRIBUTED -- predicts well, means nothing')


## 6b · Why λ cannot fix this

Closed form, no training. The loss penalises `||a_tilde||² + ||c_tilde||²`, and on
the velocity row the model must satisfy `F = r·(a0·(y/s0) + a1·(v/s1) + c)`.

Two facts fall out, and neither is tunable:

- The conditioning sandwich normalises every channel to O(1) — that is its job, and
  it is why `dA` is interpretable at all. It also makes the three channels **equally
  cheap**, so nothing in the penalty prefers the physical one.
- `||(F/3,F/3,F/3)||² < ||(F,0,0)||²`. Minimum-norm **prefers spreading the force**
  over concentrating it.

So λ does not fail to find the truth — it points away from it. Raising λ makes every
seed spread the *same* way, which reads as "identified" in section 8 and is still wrong.


In [ ]:
_ = fall.minimum_norm_split(DATA)


## 7 · Rollout

One-step error is not the quantity of interest. The floor is forward Euler driven
by the *true* derivatives at the true state, so it is the best any model could do
at this step size — a model sitting on it cannot be improved by better physics.


In [ ]:
from evaluate import evaluate

_ = evaluate(test, model, physics,
             history_path=cfg.ckpt_dir / f'{RUN_NAME}_history.json',
             fig_dir=Path('/content/figures'),
             n_pos=1, truth_name='closed form',
             pos_label='height (m)', pos_scale=1.0)


## 8 · The λ sweep (experiment 3)

Multi-seed. `pred disagr` measures agreement on the dynamics, `matrix disagr` on
the factorisation — low-and-high is the signature of a model that fits well and is
not identified.

On CADAC this was the whole story, because seed agreement was all that could be
measured. Here it is only half: run section 6 at the chosen λ to find out whether
the seeds agree **with the truth** as well as with each other.

Cheap — identifiability is structural, so it does not need long runs.


In [ ]:
RUN_SWEEP     = True                       # @param {type:"boolean"}
SWEEP_LAMBDAS = [0.0, 1e-3, 1e-2, 1e-1, 1.0]
SWEEP_SEEDS   = [0, 1, 2]
SWEEP_EPOCHS  = 40                         # @param {type:"integer"}

if RUN_SWEEP:
    from identifiability import print_table, run_sweep
    results = run_sweep(
        DATA, SWEEP_LAMBDAS, SWEEP_SEEDS, SWEEP_EPOCHS,
        Path('/content/checkpoints/sweep'),
        q_name='speed', q_threshold=40.0, vel_slice=slice(1, 2),
        buckets=fall.V_BUCKETS, modules=fall.MODULES, known=fall.DEFAULT_KNOWN,
    )
    print_table(results, len(SWEEP_SEEDS))
else:
    print('set RUN_SWEEP = True to run it')


## 9 · Truth agreement across λ

The sweep above reports seed *agreement*. This retrains one seed per λ and scores
each recovered matrix against the closed form — the plot `main` can never produce.

Slower than section 8 (one full training run per λ). Skip it if section 6 at a
single λ already answered the question.


In [ ]:
RUN_TRUTH_SWEEP = False    # @param {type:"boolean"}

if RUN_TRUTH_SWEEP:
    rows = []
    for lam in SWEEP_LAMBDAS:
        torch.manual_seed(SEED)
        m = GrayBoxSSM.from_data(train_loader, fall.make_physics(layout),
                                 n_param=len(test.param_names), hidden=HIDDEN)
        c = TrainConfig(epochs=EPOCHS, lr=LR, lambda_reg=lam,
                        ckpt_dir=Path('/content/checkpoints/truth'),
                        run_name=f'truth_lam{lam:g}', log_every=10**9,
                        device=cfg.device)
        Trainer(m, c, q_index=layout.p('speed'), vel_slice=vel,
                buckets=fall.V_BUCKETS).fit(train_loader, val_loader)
        m.to('cpu')
        s = fall.matrix_report(m, test.x, test.p)
        s['lambda'] = lam
        s['drag_share'] = s['a_vel'] / max(s['a_pos'] + s['a_vel'] + s['a_off'], 1e-12)
        rows.append(s)

    print(f"\n{'lambda':>8}{'drag share':>13}{'dA11 rel err':>15}{'force err':>12}")
    print('-' * 48)
    for r in rows:
        print(f"{r['lambda']:>8g}{100 * r['drag_share']:>12.1f}%"
              f"{100 * r['coeff_rel']:>14.2f}%{r['force_error']:>12.4f}")
else:
    print('set RUN_TRUTH_SWEEP = True to run it')


## 10 · Experiment 5 — structure instead of penalty

If the penalty cannot pick the physical factorisation, say which one it is.

`DragStructureModule` claims `dA[1,0] = 0` — *drag depends on velocity, not on
height* — via the same `exact_blocks` mechanism that freezes `dy/dt = v`. Freezing
an entry is a statement that physics determines it, and this claim qualifies as much
as the kinematic one does. `learn_delta_c=False` removes `dc[1]`, the last channel
that can carry force without going through the drag entry.

That leaves **one free entry against one equation**. No ambiguity is left to resolve,
so if the model fits at all, `dA[1,1]` *must* be `-(k/m)|v|`.

λ should now be nearly irrelevant — which is the test. Run this at `LAMBDA_REG = 0`
and compare section 6's numbers.


In [ ]:
STRUCT_LAMBDA = 0.0    # @param {type:"number"}

physics_s = fall.make_physics(layout, fall.STRUCTURED_KNOWN)
print(physics_s, '| free dA entries:', int(physics_s.free_mask().sum()))
print('free_mask =', physics_s.free_mask().tolist())

torch.manual_seed(SEED)
model_s = GrayBoxSSM.from_data(train_loader, physics_s,
                               n_param=len(test.param_names), hidden=HIDDEN,
                               learn_delta_c=False)
cfg_s = TrainConfig(epochs=EPOCHS, lr=LR, lambda_reg=STRUCT_LAMBDA,
                    ckpt_dir=Path('/content/checkpoints'),
                    run_name=f'fall_struct_lam{STRUCT_LAMBDA:g}', device=cfg.device)
t_s = Trainer(model_s, cfg_s, q_index=layout.p('speed'), vel_slice=vel,
              buckets=fall.V_BUCKETS)
t_s.fit(train_loader, val_loader)

model_s.to('cpu'); t_s.load_checkpoint(); model_s.to('cpu')
stats_s = fall.matrix_report(model_s, test.x, test.p)

print(f"\n{'':<22}{'unstructured':>15}{'structured':>13}")
print(f"{'dA[1,1] rel error':<22}{100 * stats['coeff_rel']:>14.2f}%"
      f"{100 * stats_s['coeff_rel']:>12.2f}%")
print(f"{'force error (m/s^2)':<22}{stats['force_error']:>15.4f}"
      f"{stats_s['force_error']:>13.4f}")
print(f"{'force off the drag block':<22}"
      f"{stats['a_pos'] + stats['a_off']:>15.4f}{stats_s['a_pos'] + stats_s['a_off']:>13.4f}")
